In [32]:
import os
import pandas as pd
import yaml
import random
from faker import Faker
fake = Faker('pt_BR')

def load_config(file_path: str):
    """Carrega arquivo YAML"""
    with open(file_path, "r", encoding="utf-8") as file:
        return yaml.safe_load(file)

In [33]:
# Configura o diretório de trabalho para a raiz do projeto
import os
import sys

# Define o caminho da raiz do projeto diretamente
project_root = '/home/nicolas/Projects/Personal/corporate-benefits-transactions'
os.chdir(project_root)
print(f"Diretório de trabalho atual: {os.getcwd()}")

# Adiciona o diretório raiz ao path do Python para imports
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Carrega configurações usando os novos arquivos reorganizados
business_data = load_config('./config/business_rules.yaml')       # Dados estáticos para DataFrames
parameters = load_config('./config/transaction_simulation.yaml')  # Parâmetros de configuração

print("Configurações carregadas com sucesso!")
print(f"Dados de negócio carregados: {list(business_data.keys())}")
print(f"Parâmetros de simulação carregados: {list(parameters.keys())}")

Diretório de trabalho atual: /home/nicolas/Projects/Personal/corporate-benefits-transactions
Configurações carregadas com sucesso!
Dados de negócio carregados: ['products', 'transaction_modalities', 'acquirers', 'terminal_models_by_acquirer', 'segments', 'capture_technologies', 'status_options']
Parâmetros de simulação carregados: ['data_volumes', 'approval_rates', 'generation_settings', 'usage_patterns']


### 1. Municipalities

In [5]:
# Recarregando o módulo para garantir que temos a versão corrigida
import importlib
import src.extractors.municipality_data_importer
importlib.reload(src.extractors.municipality_data_importer)
from src.extractors.municipality_data_importer import generate_municipality_data

shapefile_path = './data/raw/municipios/municipios.shp'

### 2. Products

In [9]:
def generate_products(config_files_updated):
    """Generates products DataFrame based on business rules"""
    produtos = []
    for codigo, dados in config_files_updated['products'].items():
        produto = {
            'product_id': f"PROD_{dados['code'].upper()}",
            'code': dados['code'],
            'name': dados['name'],
            'category': dados['category'],
            'description': f"Produto de {dados['name']}",
            'active': True,
            'created_at': pd.Timestamp.now()
        }
        produtos.append(produto)

    produtos_df = pd.DataFrame(produtos)
    produtos_df['code'] = produtos_df['code'].astype('category')
    produtos_df['name'] = produtos_df['name'].astype('category')
    produtos_df['category'] = produtos_df['category'].astype('category')

    return produtos_df

### 3. CNAE

In [7]:
from src.extractors.kaggle_downloader import download_kaggle_dataset

def extract_cnae_data():
    download_kaggle_dataset("diovanimerlo/cnae-brazil-2024")

    cnaes = pd.read_csv(
        './data/raw/cnae_dataset_2024.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "secao_id": str,
            "secao": "category",
            "divisao_id": str,
            "divisao": "category",
            "grupo_id": str,
            "grupo": "category",
            "classe_id": str,
            "classe": "category",
            "subclasse_id": str,
            "subclasse": "category"
        }
    )
    return cnaes

### 4. Segments

In [10]:
def generate_segments(config_files_updated):
    """Generate segments DataFrame from business rules"""
    segments_list = config_files_updated['segments']
    segments = pd.DataFrame(segments_list)
    segments['created_at'] = pd.Timestamp.now()
    segments['name'] = segments['name'].astype('category')
    segments['category'] = segments['category'].astype('category')
    return segments


### 5. Acquirers

In [14]:
def generate_acquirers(config_files_updated):
    """Generates DataFrame for Acquirers based on business rules"""
    acquirers = []
    for idx, name in enumerate(config_files_updated['acquirers'], start=1):
        acquirer = {
            "acquirer_id": f"ACQ_{idx:03d}",
            "name": name,
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        acquirers.append(acquirer)
    return pd.DataFrame(acquirers)

### 6. Transaction Modalities

In [13]:
def generate_transaction_modalities(config_files_updated):
    """Generates DataFrame for Transaction Modalities based on business rules"""
    transaction_modalities = []
    modalidades_yaml = config_files_updated['transaction_modalities']

    # Se for dict, usa .values(); se for lista, itera direto
    iterator = modalidades_yaml.values() if isinstance(modalidades_yaml, dict) else modalidades_yaml

    for idx, dados in enumerate(iterator, start=1):
        modality = {
            "modality_id": f"MOD_{idx:03d}",
            "code": dados["name"].upper().replace(" ", "_"),
            "name": dados["name"],
            "type": "brand" if dados["name"].upper() in ["ELO", "MASTERCARD", "VISA"] else "proprietary_network",
            "active": True,
            "created_at": pd.Timestamp.now()
        }
        transaction_modalities.append(modality)
    return pd.DataFrame(transaction_modalities)

### 7. MCC Categories

In [12]:
import pandas as pd

def extract_mcc_data():
    mcc = pd.read_csv(
        './data/raw/mcc.csv',
        sep=',',
        encoding='utf-8',
        dtype={
            "mcc_id": str,
            "mcc_code": str,
            "mcc_description": str,
            "valid_payment_brands": str,
            "category": "category"
        }
    )
    return mcc

### 8. Capture Technologies

In [11]:
def generate_capture_technologies():
    """Generate capture technologies"""
    technologies = [
        {"code": "CHIP", "category": "physical", "name": "Chip", "requires_physical_card": True},
        {"code": "MAG_STRIPE", "category": "physical", "name": "Magnetic Stripe", "requires_physical_card": True},
        {"code": "CONTACTLESS", "category": "physical", "name": "Contactless", "requires_physical_card": True},
        {"code": "MOBILE_NFC", "category": "digital", "name": "Mobile NFC", "requires_physical_card": False},
        {"code": "APPLE_PAY", "category": "digital", "name": "Apple Pay", "requires_physical_card": False},
        {"code": "GOOGLE_PAY", "category": "digital", "name": "Google Pay", "requires_physical_card": False}
    ]

    capture_tech_list = []
    for idx, tech in enumerate(technologies, start=1):
        capture_tech = {
            "technology_id": f"TECH_{idx:03d}",
            "technology_code": tech["code"],
            "technology_category": tech["category"],
            "technology_name": tech["name"],
            "requires_physical_card": tech["requires_physical_card"],
            "created_at": pd.Timestamp.now()
        }
        capture_tech_list.append(capture_tech)

    return pd.DataFrame(capture_tech_list)

### 9. Establishments 

In [41]:
def generate_establishments(n_establishments, df_cnae, df_mcc, df_municipalities):
    """Generate establishments DataFrame based on existing CNAE, MCC and municipalities data"""
    establishments = []
    
    # Tipos de natureza jurídica mais comuns
    legal_natures = ["LTDA", "EIRELI", "SA", "MEI", "EPP", "COOPERATIVA"]
    company_sizes = ["MEI", "ME", "EPP", "PEQUENO PORTE", "MEDIO PORTE", "GRANDE PORTE"]
    statuses = ["ATIVA", "INATIVA", "SUSPENSA", "BAIXADA"]
    status_reasons = ["NORMAL", "IRREGULARIDADE", "FALTA DE MOVIMENTO", "MUDANÇA DE ENDEREÇO"]
    
    for i in range(n_establishments):
        establishment = {
            'establishment_id': f"EST_{i:06d}",
            'cnpj': fake.cnpj(),
            'legal_name': fake.company(),
            'trade_name': fake.company_suffix() + " " + fake.last_name(),
            'main_cnae_id': random.choice(df_cnae['subclasse_id'].tolist()),
            'mcc_code': random.choice(df_mcc['mcc_code'].tolist()),
            'legal_nature': random.choice(legal_natures),
            'company_size': random.choice(company_sizes),
            'municipality_id': random.choice(df_municipalities['municipality_id'].tolist()),
            'postal_code': fake.postcode(),
            'street': fake.street_name(),
            'number': str(fake.building_number()),
            'complement': random.choice([None, "Sala " + str(random.randint(1, 100)), "Loja " + str(random.randint(1, 50))]),
            'neighborhood': fake.neighborhood(),
            'email': fake.company_email(),
            'phone': fake.phone_number(),
            'registration_status': random.choice(statuses),
            'status_date': fake.date_between(start_date='-2y', end_date='today'),
            'status_reason': random.choice(status_reasons),
            'opening_date': fake.date_between(start_date='-10y', end_date='-1y'),
            'created_at': pd.Timestamp.now()
        }
        establishments.append(establishment)
    
    return pd.DataFrame(establishments)

### 10. Companies

In [16]:
def generate_companies(n_companies, municipalities, segments):
    companies = []
    for i in range(n_companies):
        company = {
            'company_id': f"EMP_{i}",
            'parent_company_id': random.choice([None] + [c["company_id"] for c in companies]) if companies else None,
            'name': fake.company(),
            'cnpj': fake.cnpj(),
            'company_type': random.choice(["Privada", "Publica"]),
            'segment_id': random.choice(segments['segment_id'].tolist()),
            'municipality_id': random.choice(municipalities['municipality_id'].tolist()),
            'contract_registration_date': fake.date_between(start_date='-5y', end_date='today'),
            'created_at': pd.Timestamp.now()
        }
        companies.append(company)
    return pd.DataFrame(companies)  # Return as DataFrame for consistency

### 11. Cardholders

In [17]:
def generate_cardholders(n_funcionarios, companies_df, municipalities_df):
    """Generate cardholders DataFrame based on existing companies data"""
    cardholders = []
    
    for i in range(n_funcionarios):
        cardholder = {
            'cardholder_id': f"FUNC_{i}",
            'company_id': random.choice(companies_df['company_id'].tolist()),
            'name': fake.name(),
            'cpf': fake.cpf(),
            'municipality_id': random.choice(municipalities_df['municipality_id'].tolist()),
            'admission_date': fake.date_this_decade(),
            'position': fake.job(),
            'department': fake.word(),
            'status': random.choice(['active', 'inactive', 'terminated']),
            'created_at': pd.Timestamp.now()
        }
        cardholders.append(cardholder)
    
    return pd.DataFrame(cardholders)

### 12. Cards

In [19]:
def generate_cards(n_cards, cardholders_df, modality_df):
    cards = []

    for idx in range(n_cards):
        card_number = "".join([str(random.randint(0, 9)) for _ in range(16)])
        formatted_card_number = " ".join([card_number[i:i+4] for i in range(0, 16, 4)])
        
        cvv = "".join([str(random.randint(0, 9)) for _ in range(3)])

        card = {
            "card_id": idx,
            "cardholder_id": random.choice(cardholders_df['cardholder_id'].tolist()),
            "card_number": formatted_card_number,
            "modality_id": random.choice(modality_df['modality_id'].tolist()),
            "expiration_date": fake.date_between(start_date='today', end_date='+5y'),
            "cvv": cvv,
            "card_type": random.choice(['credit', 'debit']),
            "issuance_date": pd.Timestamp.now(),
            "status": random.choice(['active', 'inactive']),
            "created_at": pd.Timestamp.now()
        }
        cards.append(card)

    return pd.DataFrame(cards)

### 13. Card Products (TODO)

In [20]:
def generate_card_products(cards_df, products_df, min_products=1, max_products=3):
    """
    Gera produtos associados aos cartões.
    Cada cartão pode ter entre min_products e max_products produtos.
    """
    card_products = []

    for card_id in cards_df['card_id'].tolist():
        n_products = random.randint(min_products, max_products)

        for _ in range(n_products):
            activation_date = fake.date_between(start_date='-2y', end_date='today')

            # 30% de chance de já ter sido desativado
            if random.random() < 0.3:
                deactivation_date = fake.date_between(start_date=activation_date, end_date='today')
                status = random.choice(['suspended', 'cancelled'])
            else:
                deactivation_date = None
                status = 'active'

            card_product = {
                "card_product_id": fake.uuid4(),
                "card_id": card_id,
                "product_id": random.choice(products_df['product_id'].tolist()),
                "current_balance": round(random.uniform(0, 5000), 2),
                "daily_limit": round(random.uniform(100, 2000), 2),
                "monthly_limit": round(random.uniform(1000, 10000), 2),
                "activation_date": activation_date,
                "deactivation_date": deactivation_date,
                "status": status,
                "created_at": pd.Timestamp.now()
            }
            card_products.append(card_product)

    return pd.DataFrame(card_products)


### 14.Terminals

In [42]:
def generate_terminals(n_terminals, establishments_df, acquirers_df, business_data):
    """Generate terminals for establishments with correct model/acquirer/type relation"""
    terminals = []

    models_by_acquirer = business_data["terminal_models_by_acquirer"]
    # Mapeamento modelo -> tipo
    model_type_map = {
        "POS Cielo LIO": "POS",
        "TEF Rede": "TEF",
        "Stone Ton": "POS",
        "GetNet POS": "POS",
        "Moderninha Pro": "POS",
        "SafraPay POS": "POS",
        "POS Genérico": "POS"
    }

    for i in range(n_terminals):
        acquirer_name = random.choice(acquirers_df['name'].tolist())
        terminal_model = random.choice(models_by_acquirer.get(acquirer_name, ["POS Genérico"]))
        terminal_type = model_type_map.get(terminal_model, "POS")
        terminal = {
            'terminal_id': f"TERM_{i:08d}",
            'terminal_code': f"T{random.randint(100000, 999999)}",
            'establishment_id': random.choice(establishments_df['establishment_id'].tolist()),
            'acquirer_id': acquirers_df[acquirers_df['name'] == acquirer_name]['acquirer_id'].iloc[0],
            'terminal_model': terminal_model,
            'terminal_type': terminal_type,
            'status': random.choice(['active', 'inactive', 'maintenance']),
            'installation_date': fake.date_between(start_date='-3y', end_date='today'),
            'created_at': pd.Timestamp.now()
        }
        terminals.append(terminal)

    return pd.DataFrame(terminals)

### Generate Response Codes

In [22]:
def generate_response_codes():
    """Generate response codes for transaction processing"""
    response_codes = [
        {"code": "00", "description": "Approved", "category": "success"},
        {"code": "51", "description": "Insufficient funds", "category": "balance"},
        {"code": "14", "description": "Invalid card number", "category": "card"},
        {"code": "55", "description": "Incorrect PIN", "category": "password"},
        {"code": "96", "description": "System error", "category": "network"},
        {"code": "30", "description": "Format error", "category": "network"},
        {"code": "12", "description": "Invalid transaction", "category": "establishment"},
        {"code": "43", "description": "Stolen card", "category": "card"}
    ]

    codes_list = []
    for idx, code_info in enumerate(response_codes, start=1):
        response_code = {
            "response_code_id": f"RESP_{idx:03d}",
            "response_code": code_info["code"],
            "response_description": code_info["description"],
            "response_category": code_info["category"],
            "created_at": pd.Timestamp.now()
        }
        codes_list.append(response_code)

    return pd.DataFrame(codes_list)


###  DataFrames

In [43]:
municipalities_df = generate_municipality_data(shapefile_path)
products_df = generate_products(business_data)
cnaes_df = extract_cnae_data()
segments_df = generate_segments(business_data)
acquirers_df = generate_acquirers(business_data)
transaction_modalities_df = generate_transaction_modalities(business_data)
mcc_df = extract_mcc_data()
capture_tech_df = generate_capture_technologies()
establishments_df = generate_establishments(parameters['data_volumes']['establishments'], cnaes_df, mcc_df, municipalities_df)  # <- sem vírgula
terminals_df = generate_terminals(parameters['data_volumes']['terminals'], establishments_df, acquirers_df, business_data)      # <- sem vírgula
companies_df = generate_companies(parameters['data_volumes']['companies'], municipalities_df, segments_df)
cardholders_df = generate_cardholders(parameters['data_volumes']['cardholders'], companies_df, municipalities_df)
cards_df = generate_cards(parameters['data_volumes']['cards'], cardholders_df, transaction_modalities_df)
card_products_df = generate_card_products(cards_df, products_df, min_products=1, max_products=3)

Path to dataset files: /home/nicolas/.cache/kagglehub/datasets/diovanimerlo/cnae-brazil-2024/versions/1
Arquivo já existe, pulando: /home/nicolas/Projects/Personal/corporate-benefits-transactions/src/extractors/../../data/raw/cnae_dataset_2024.csv


In [24]:
import numpy as np

def gerar_valor_transacao(max_valor=1000, escala=250):
    """
    Gera um valor aleatório progressivo para transação.
    Quanto maior o valor, menor a chance de ocorrer.
    """
    valor = np.random.exponential(scale=escala)
    return round(min(valor, max_valor), 2)

## OLTP process

## 📊 PROCESSO OLTP - Transformação para Estrutura OLTP

Agora vamos transformar os dados gerados para seguir a estrutura das tabelas OLTP, sem ainda inserir no banco.

### Gera Transações

In [46]:
def generate_transactions(parameters, cards_df, establishments_df, terminals_df, card_products_df):
    """
    Gera transações simples para simular operação OLTP
    Usa os dados já criados (cards, establishments, etc.)
    """
    transactions = []

    # Só estabelecimentos que têm pelo menos um terminal
    valid_establishments = terminals_df['establishment_id'].unique()

    for i in range(parameters['data_volumes']['transactions']):
        card_id = cards_df.sample(1)['card_id'].iloc[0]
        establishment_id = random.choice(valid_establishments)
        terminals_for_est = terminals_df[terminals_df['establishment_id'] == establishment_id]
        terminal_id = terminals_for_est['terminal_id'].sample(1).iloc[0]
        product_id = card_products_df[card_products_df['card_id'] == card_id]['product_id'].sample(1).iloc[0]
        amount = gerar_valor_transacao(max_valor=500, escala=100)

        transaction = {
            "transaction_id": fake.uuid4(),
            "card_id": card_id,
            "product_id": product_id,
            "establishment_id": establishment_id,
            "terminal_id": terminal_id,
            "amount": amount,
            "date": fake.date_time_between(start_date='-30d', end_date='now'),
            "capture_type": random.choice(['chip', 'magnetic_stripe', 'contactless', 'nfc']),
            "response_code": None,
            "response_message": None,
            "created_at": pd.Timestamp.now(),
            "updated_at": pd.Timestamp.now()
        }
        transactions.append(transaction)

    return pd.DataFrame(transactions)

transactions_df = generate_transactions(parameters, cards_df, establishments_df, terminals_df, card_products_df)

### Autorizador de transações

In [ ]:
def generate_authorizations(transactions_df):
    """
    Autorizador simples que processa transações e gera autorizações
    Simula aprovações/negações baseado em regras simples
    """
    authorizations = []
    
    for _, transaction in transactions_df.iterrows():
        # 85% de aprovação, 15% de negação
        is_approved = random.random() < 0.85
        
        if is_approved:
            response_code = "00"  # Approved
            response_message = "Aprovado"
            status = "approved"
            auth_code = f"AUTH{random.randint(100000, 999999)}"
        else:
            # Códigos de negação mais comuns
            denial_codes = [
                ("51", "Saldo insuficiente", "denied"),
                ("14", "Número do cartão inválido", "denied"),
                ("55", "PIN incorreto", "denied"),
                ("43", "Cartão bloqueado", "denied"),
                ("96", "Erro do sistema", "system_error")
            ]
            response_code, response_message, status = random.choice(denial_codes)
            auth_code = None
        
        authorization = {
            "authorization_id": fake.uuid4(),
            "transaction_id": transaction["transaction_id"],
            "authorization_code": auth_code,
            "nsu": f"NSU{random.randint(100000000, 999999999)}",
            "authorization_datetime": transaction["date"],
            "status": status,
            "response_code": response_code,
            "response_message": response_message,
            "created_at": pd.Timestamp.now()
        }
        authorizations.append(authorization)
    
    return pd.DataFrame(authorizations)

def update_transactions_with_authorization(transactions_df, authorizations_df):
    """
    Atualiza as transações com os códigos de resposta do autorizador
    """
    # Criar um merge baseado no transaction_id
    updated_transactions = transactions_df.merge(
        authorizations_df[['transaction_id', 'response_code', 'response_message']], 
        on='transaction_id', 
        how='left'
    )
    
    # Atualizar os campos que estavam None
    updated_transactions['response_code'] = updated_transactions['response_code_y']
    updated_transactions['response_message'] = updated_transactions['response_message_y']
    updated_transactions['updated_at'] = pd.Timestamp.now()
    
    # Remover colunas duplicadas do merge
    updated_transactions = updated_transactions.drop(columns=['response_code_x', 'response_code_y', 
                                                              'response_message_x', 'response_message_y'])
    
    return updated_transactions

# Processar transações
authorizations_df = generate_authorizations(transactions_df)
transactions_df = update_transactions_with_authorization(transactions_df, authorizations_df)